# SKU Assignment Pipeline — RMFS Replenishment

Follows Rika's baseline: ABC classification from `items_dictionary` drives pod assignment priority.  
No demand-feature clustering — classification is already in the data.

| Phase | Step | Output |
|-------|------|--------|
| 1 | Item-Pod Configuration | `items_slots_configuration.csv`, `best_config_per_sku.csv` |
| 2 | Prepare SKU table (ABC → cluster int) | `sku_sample.csv` |
| 3 | SKU-to-Pod Assignment (FFD) | `sku_assignment_detail.csv`, `sku_assignment.csv`, `pod_summary.csv` |
| 4 | Convert to Simulation Format | `netlogo/items.csv`, `netlogo/pods.csv` |
| 5 | Run Simulation (Baseline) | `results_baseline/` |
| 6 | Results & Analysis | plots, tables |

## Setup

In [ ]:
import sys
import os
from pathlib import Path

# Repo root — adjust if notebook is not in the project root
REPO = Path.cwd()
# REPO = Path("/Users/brendantm/Taiwan/TEEP/rmfs-replenishment")

for _p in [REPO, REPO / "assignment", REPO / "pipeline"]:
    p = str(_p)
    if p not in sys.path:
        sys.path.insert(0, p)

os.chdir(REPO)
print("Working directory:", Path.cwd())

---
## Phase 1 — Item-Pod Configuration

Generates feasible `(item, slot_type, orientation)` rows from box dimensions and slot sizes.

In [ ]:
from item_pod_config import load_pod_size, load_items, generate_configs, pick_best_config

slot_df  = load_pod_size(REPO / "pod_size.csv")
items_df = load_items(REPO / "sku_sample.csv")

print(f"Slot types : {sorted(slot_df['slot_type'].astype(int).tolist())}")
print(f"SKUs       : {len(items_df)}")

In [ ]:
config_df = generate_configs(items_df, slot_df)

covered   = config_df["item_code"].nunique()
uncovered = len(items_df) - covered
print(f"Total config rows : {len(config_df)}")
print(f"SKUs covered      : {covered}")
print(f"SKUs uncovered    : {uncovered}")

config_df.to_csv(REPO / "items_slots_configuration.csv", index=False)

best_df = pick_best_config(config_df)
best_df.to_csv(REPO / "best_config_per_sku.csv", index=False)

print("Saved items_slots_configuration.csv and best_config_per_sku.csv")

config_df.groupby("slot_type").agg(
    rows=("item_code", "count"),
    skus=("item_code", "nunique"),
    avg_max_boxes=("max_boxes_in_slot", "mean"),
    avg_vol_util=("vol_util_slot", "mean"),
)

---
## Phase 2 — Prepare SKU Table

Reads `sku_sample.csv` (k-means cluster label already present).  
Filters to feasible SKUs (those that fit at least one slot type) and writes the result.

Cluster priority used in assignment (`CLUSTER_PRIORITY = [3, 1, 0, 2]` in `sku_assignment.py`):  
Cluster 3 (high demand) → processed first, Cluster 1 → second, Cluster 0 → third, Cluster 2 → last.

In [ ]:
import pandas as pd
import numpy as np

sku_raw = pd.read_csv(REPO / "sku_sample.csv", dtype={"item_code": str})

# Only keep SKUs that fit at least one slot type
feasible_codes = set(config_df["item_code"].astype(str).unique())
sku_sample = sku_raw[sku_raw["item_code"].isin(feasible_codes)].copy().reset_index(drop=True)

sku_sample["cluster"] = sku_sample["cluster"].fillna(0).astype(int)

sku_sample.to_csv(REPO / "sku_sample.csv", index=False)

print(f"SKUs in sku_sample.csv: {len(sku_sample)}")
print(sku_sample["cluster"].value_counts().sort_index())

---
## Phase 3 — SKU-to-Pod Assignment (FFD)

Greedy FFD algorithm: constrained items first (≤2 compatible slot types), then flexible items, then secondary fill of empty slots.  
Outputs: `sku_assignment_detail.csv`, `sku_assignment.csv`, `pod_summary.csv`, `unassigned_skus.csv`.

In [ ]:
from sku_assignment import run_assignment

detail_df = run_assignment(
    base_dir                = REPO,
    sku_sample_path         = REPO / "sku_sample.csv",
    items_dict_path         = REPO / "items_dictionary_cleaned.csv",
    items_slots_config_path = REPO / "items_slots_configuration.csv",
)
print(f"Assignment rows: {len(detail_df)}  |  Pods used: {detail_df['pod_id'].nunique()}")

### Assignment Results

In [ ]:
detail_df = pd.read_csv(REPO / "sku_assignment_detail.csv")
print(f"Rows: {len(detail_df)}  |  Pods used: {detail_df['pod_id'].nunique()}")
detail_df.head(10)

In [ ]:
sku_df = pd.read_csv(REPO / "sku_assignment.csv")
print(f"SKUs assigned: {len(sku_df)}")
sku_df.head(10)

In [ ]:
import matplotlib.pyplot as plt

pod_df = pd.read_csv(REPO / "pod_summary.csv")
print(f"Total pods: {len(pod_df)}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(pod_df["total_weight_kg"], bins=30, edgecolor="black")
axes[0].axvline(1300, color="red", linestyle="--", label="Max weight (1300 kg)")
axes[0].set_title("Pod Weight Distribution"); axes[0].set_xlabel("Weight (kg)"); axes[0].legend()
axes[1].hist(pod_df["slots_used"], bins=range(0, 20), edgecolor="black")
axes[1].set_title("Slots Used per Pod"); axes[1].set_xlabel("Slots used")
plt.tight_layout(); plt.show()

pod_df.describe()

In [ ]:
# Assignment by cluster
if "cluster" in sku_df.columns:
    cs = sku_df.groupby("cluster").agg(
        n_skus=("item_code", "count"),
        boxes_assigned=("boxes_assigned", "sum"),
        boxes_unassigned=("boxes_unassigned", "sum"),
    )
    cs["pct_assigned"] = (
        cs["boxes_assigned"] / (cs["boxes_assigned"] + cs["boxes_unassigned"]) * 100
    ).round(1)
    print(cs)

In [ ]:
unassigned_path = REPO / "unassigned_skus.csv"
if unassigned_path.exists():
    unassigned_df = pd.read_csv(unassigned_path)
    print(f"Unassigned SKUs: {len(unassigned_df)}")
    unassigned_df.head(20)
else:
    print("No unassigned_skus.csv — all SKUs fully assigned!")

---
## Phase 4 — Convert to Simulation Format

Translates the FFD assignment into the NetLogo simulator's `items.csv` and `pods.csv` formats.

In [ ]:
from convert_to_sim import generate_items_csv, convert_detail_to_pods_csv

netlogo_dir = REPO / "netlogo"

items_sim = generate_items_csv(
    sku_sample_path = REPO / "sku_sample.csv",
    items_dict_path = REPO / "items_dictionary_cleaned.csv",
    output_path     = netlogo_dir / "items.csv",
)

pods_sim = convert_detail_to_pods_csv(
    detail_df      = detail_df,
    items_df       = items_sim,
    output_path    = netlogo_dir / "pods.csv",
    pods_dict_path = netlogo_dir / "pods_dictionary.csv",
)

print(f"items.csv : {len(items_sim)} items")
print(f"pods.csv  : {len(pods_sim)} slot assignments across {pods_sim['pod_id'].nunique()} pods")

---
## Phase 5 — Run Simulation (Baseline)

Output: `results_baseline/summary.csv`, `tick_metrics.csv`, `orders.csv`.

> Set `MAX_TICKS` lower (e.g. 500) for a quick smoke-test.

In [ ]:
import subprocess

MAX_TICKS = 1000

proc = subprocess.run(
    [sys.executable, str(REPO / "pipeline" / "run_baseline.py"),
     "--max-ticks", str(MAX_TICKS)],
    text=True,
    capture_output=True,
)
print(proc.stderr)
if proc.stdout:
    print(proc.stdout)
if proc.returncode != 0:
    print(f"[ERROR] process exited with code {proc.returncode}")

---
## Phase 6 — Results & Analysis

In [ ]:
results_dir = REPO / "results_baseline"

summary_path = results_dir / "summary.csv"
if summary_path.exists():
    summary = pd.read_csv(summary_path)
    print(summary.T.to_string())
else:
    print("No summary.csv yet — run Phase 5 first.")

In [ ]:
tick_path = results_dir / "tick_metrics.csv"
if tick_path.exists():
    tick_df = pd.read_csv(tick_path)
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    for ax, (col, title) in zip(axes.flat, [
        ("total_energy",  "Total Energy"),
        ("job_queue_len", "Job Queue Length"),
        ("stop_and_go",   "Stop & Go Events"),
        ("total_turning", "Total Turning"),
    ]):
        if col in tick_df.columns:
            ax.plot(tick_df["tick"], tick_df[col])
            ax.set_title(title); ax.set_xlabel("Tick")
    plt.suptitle("Simulation Tick Metrics — Baseline", fontsize=13)
    plt.tight_layout(); plt.show()
else:
    print("No tick_metrics.csv yet — run Phase 5 first.")

In [ ]:
orders_path = results_dir / "orders.csv"
if orders_path.exists():
    orders_df = pd.read_csv(orders_path)
    if "order_complete_time" in orders_df.columns and "process_start_time" in orders_df.columns:
        orders_df["cycle_time"] = orders_df["order_complete_time"] - orders_df["process_start_time"]
        print(orders_df["cycle_time"].describe())
        plt.figure(figsize=(8, 4))
        plt.hist(orders_df["cycle_time"].dropna(), bins=40, edgecolor="black")
        plt.title("Order Cycle Time Distribution"); plt.xlabel("Cycle time (ticks)")
        plt.tight_layout(); plt.show()
    else:
        print(orders_df.head())
else:
    print("No orders.csv yet — run Phase 5 first.")